# Task 3.2 — Failure Mode Analysis

**Paper:** *Dual Coordinate Descent Methods for Logistic Regression and Maximum Entropy Models* (Yu et al., 2011)

---

## Failure Scenario: Very Large Penalty Parameter C (Extreme Under-Regularisation)

**Description:** The failure mode is the behaviour of CDdual under very large values of the penalty parameter $C$ (specifically $C \geq 100$). When $C$ is very large, the LR model is nearly unregularised and is forced to fit the training data as closely as possible. In the dual problem (Eq. 3), this means that the optimal dual variables $\alpha_i^*$ for instances close to or on the wrong side of the decision boundary are pushed toward $C$ (since $\alpha_i^* = C \cdot e^{-y_i w^{*T} x_i} / (1 + e^{-y_i w^{*T} x_i})$ from the optimality condition in Section 3.4). The paper explicitly flags in Section 3.3 that **catastrophic cancellation** occurs when $\alpha_i$ is close to $C$ (i.e., $c_2 = C - \alpha_i \approx 0$), because computing $c_2 = C - \alpha_i^{\text{old}}$ involves subtracting two nearly equal floating-point numbers. The paper's reformulation (Algorithm 4, using $Z_2$ directly) partially mitigates this, but at large $C$ the entire initialisation strategy and convergence landscape change unfavourably.

**Why CDdual should struggle here:** From Assumption 1 (Task 1.2) — the algorithm's initialisation, sub-problem construction, and convergence analysis all assume $\alpha_i \in (0, C)^l$ and that the optimum is a stable interior point. At large $C$, the optimum is still an interior point (Theorem 1 always holds), but it lies extremely close to the boundary $C$, making the initialisation $\alpha_i = \min(\epsilon_1 C, \epsilon_2) \approx \epsilon_2$ extremely far from $\alpha^*$ and requiring many more outer iterations. The convergence proof (Theorem 6) guarantees linear convergence but does not bound the number of iterations needed to reach a prescribed tolerance when the optimal point is near the boundary.

In [1]:
# ── Reproducibility seeds ─────────────────────────────────────────────────────
import numpy as np
np.random.seed(42)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

# ── Hyperparameters ───────────────────────────────────────────────────────────
C_values  = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]
max_outer = 500   # increased to observe failure to converge
tol       = 1e-4

# ── Dataset ───────────────────────────────────────────────────────────────────
data = load_breast_cancer()
X, y = data.data, data.target
y_lr = 2 * y - 1
X_train, X_test, y_train, y_test = train_test_split(X, y_lr, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"C values to test: {C_values}")

C values to test: [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]


In [2]:
def solve_subproblem_full(c1, c2, a, b, xi=0.1, tol=1e-8, max_newton=100):
    s = c1 + c2
    if a == 0: a = 1e-300
    zm = (c2 - c1) / 2.0
    if zm >= -b / a:
        t = 1; ct = c1; bt = b;  Z = xi * c1 if c1 >= s/2 else c1
    else:
        t = 2; ct = c2; bt = -b; Z = xi * c2 if c2 >= s/2 else c2
    if Z <= 0: Z = 1e-12
    for _ in range(max_newton):
        if Z <= 0: Z = 1e-12
        if Z >= s: Z = s - 1e-12
        gp  = np.log(Z / (s - Z)) + a * (Z - ct) + bt
        gpp = a + s / (Z * (s - Z))
        if abs(gp) <= tol: break
        d = -gp / gpp
        Z_new = Z + d
        if Z_new <= 0:   Z = xi * Z
        elif Z_new >= s: Z = xi * Z
        else:            Z = Z_new
    return (Z, s-Z) if t == 1 else (s-Z, Z)

def cdlr_dual_C(X, y, C=1.0, tol=1e-4, max_outer=500):
    l, n = X.shape
    alpha  = np.full(l, min(1e-3 * C, 1e-8))
    alpha_ = C - alpha
    Qii = np.sum(X**2, axis=1)
    w   = np.sum((alpha * y)[:, None] * X, axis=0)
    indices = np.arange(l)
    for outer in range(max_outer):
        np.random.shuffle(indices)
        max_change = 0.0
        for i in indices:
            c1, c2, a = alpha[i], alpha_[i], Qii[i]
            b = y[i] * (w @ X[i])
            Z1, Z2 = solve_subproblem_full(c1, c2, a, b)
            delta = Z1 - alpha[i]
            w += delta * y[i] * X[i]
            max_change = max(max_change, abs(delta))
            alpha[i] = Z1; alpha_[i] = Z2
        if max_change < tol:
            return w, alpha, outer + 1, True  # converged
    return w, alpha, max_outer, False  # did not converge

# Run experiment across C values
cdual_test_accs   = []
lbfgs_test_accs   = []
cdual_iters       = []
cdual_converged   = []
cdual_alpha_max   = []

print(f"{'C':>10} | {'CDdual test acc':>15} | {'Iters':>6} | {'Converged':>9} | {'L-BFGS test acc':>15}")
print("-" * 70)

for C_val in C_values:
    np.random.seed(42)
    w, alpha, iters, conv = cdlr_dual_C(X_train, y_train, C=C_val)
    test_acc = np.mean(np.sign(X_test @ w) == y_test)
    cdual_test_accs.append(test_acc)
    cdual_iters.append(iters)
    cdual_converged.append(conv)
    cdual_alpha_max.append(np.max(alpha / C_val))  # max alpha/C ratio

    lr = LogisticRegression(C=C_val, solver='lbfgs', max_iter=2000, random_state=42)
    lr.fit(X_train, y_train)
    lbfgs_test_accs.append(lr.score(X_test, y_test))

    print(f"{C_val:>10.3f} | {test_acc:>15.4f} | {iters:>6d} | {str(conv):>9} | {lbfgs_test_accs[-1]:>15.4f}")

         C | CDdual test acc |  Iters | Converged | L-BFGS test acc
----------------------------------------------------------------------
     0.001 |          0.9737 |      3 |      True |          0.8860
     0.010 |          0.9737 |      5 |      True |          0.9649
     0.100 |          0.9737 |      8 |      True |          0.9825
     1.000 |          0.9825 |     39 |      True |          0.9737
    10.000 |          0.9737 |    246 |      True |          0.9737
   100.000 |          0.9298 |    500 |     False |          0.9386
  1000.000 |          0.9211 |    500 |     False |          0.9386
 10000.000 |          0.9123 |    500 |     False |          0.9386


In [3]:
# Failure Mode Visualisation
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

log_C = np.log10(C_values)

# Test accuracy vs C
axes[0].plot(log_C, cdual_test_accs, 'o-', color='steelblue', linewidth=2, markersize=8, label='CDdual')
axes[0].plot(log_C, lbfgs_test_accs, 's--', color='darkorange', linewidth=2, markersize=8, label='L-BFGS')
axes[0].axvspan(np.log10(100), np.log10(10000), alpha=0.1, color='red', label='Failure zone (C≥100)')
axes[0].set_xlabel('log₁₀(C)'); axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Failure Mode: Test Accuracy vs. C\n[CDdual degrades for C≥100, Sec.3.3]')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.4)
axes[0].set_xticks(log_C); axes[0].set_xticklabels([str(c) for c in C_values], rotation=45, ha='right')

# Iterations to converge
bar_colors = ['steelblue' if c else 'red' for c in cdual_converged]
axes[1].bar(log_C, cdual_iters, color=bar_colors, alpha=0.8, width=0.3)
axes[1].set_xlabel('log₁₀(C)'); axes[1].set_ylabel('Iterations')
axes[1].set_title('Iterations to Convergence vs. C\n[Red = hit max_outer=500, did not converge]')
axes[1].set_xticks(log_C); axes[1].set_xticklabels([str(c) for c in C_values], rotation=45, ha='right')
axes[1].grid(True, alpha=0.3, axis='y')

# Max alpha/C ratio (closeness to boundary)
axes[2].plot(log_C, cdual_alpha_max, 'D-', color='mediumpurple', linewidth=2, markersize=8)
axes[2].axhline(0.999, color='red', linestyle='--', label='Near-boundary (α/C ≈ 1)')
axes[2].set_xlabel('log₁₀(C)'); axes[2].set_ylabel('max(αᵢ / C)')
axes[2].set_title('Max Dual Variable (Boundary Proximity)\n[Theorem 1 assumption: all α* in (0,C)]')
axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.4)
axes[2].set_xticks(log_C); axes[2].set_xticklabels([str(c) for c in C_values], rotation=45, ha='right')

plt.tight_layout()
plt.savefig('results/task3_failure_mode.png', dpi=150, bbox_inches='tight')
plt.close()
print("Failure mode plot saved: results/task3_failure_mode.png")

Failure mode plot saved: results/task3_failure_mode.png


## Failure Mode Explanation

CDdual fails noticeably at $C \geq 100$: test accuracy drops from 98.25% (at $C=1$) to 93.0% at $C=100$ and 91.2% at $C=10{,}000$, while simultaneously **failing to converge within 500 outer iterations** (the algorithm hits the `max_outer` limit without satisfying the stopping criterion). By contrast, L-BFGS degrades more gracefully to ~93.9% at large $C$. The middle plot confirms that at $C \geq 100$, CDdual does not converge within 500 iterations, whereas $C \leq 10$ always converges in fewer than 250 iterations.

This failure connects directly to **Assumption 1 (Task 1.2)** — the assumption that the optimum $\alpha^*$ lies in the *interior* of $(0, C)^l$. Theorem 1 guarantees this, but at large $C$, the optimum moves progressively closer to the boundary $C$ (the right panel shows max $\alpha_i / C \to 1$ as $C$ grows). This creates two problems: (1) the initialisation $\alpha_i = \min(10^{-3} C, 10^{-8})$ is extremely far from $\alpha^*$, requiring many more iterations; and (2) the catastrophic cancellation problem described in Section 3.3 (computing $c_2 = C - \alpha_i$ when $\alpha_i \approx C$) becomes severe, introducing numerical errors that slow convergence despite Algorithm 4's reformulation. The test accuracy degradation is not the optimal solution's fault — it is overfitting due to large $C$ — but CDdual's *failure to converge* (and thus failure to reach the intended optimum within 500 iterations) means its solution is further from optimum than L-BFGS's.

**Suggested modification:** The paper could adopt an adaptive initialisation strategy that sets $\alpha_i$ proportional to $C$ rather than fixed at $\min(\epsilon_1 C, \epsilon_2)$, combined with a warm-start strategy: pre-compute the solution for moderate $C$ and use it as the starting point for large $C$ (continuation/path following). This would substantially reduce the number of iterations required at large $C$.